# 03 Feature Engineering

This notebook focuses on the feature engineering pipeline for the Sentinel Sepsis-Associated AKI (SA-AKI) Early Warning System.

## Strategy
- **Missingness Indicators**: Missing data in EHR is highly informative. We explicitly create missingness flags before imputation.
- **Rolling Windows**: To capture trends in vital signs and lab results, we compute rolling aggregates (mean, max, min, std) over time windows (e.g., 6h, 12h, 24h).
- **Creatinine Velocity**: The rate of change in serum creatinine is a critical physiological marker for acute kidney injury.
- **Composite Features**: Combining related measurements (e.g., BUN-to-creatinine ratio, shock index).

In [ ]:
# Colab setup: Uncomment if running in Google Colab
# !pip install -q pandas numpy
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/sentinel-poc/project_sentinel

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
import os
sys.path.append(os.path.abspath('..'))

from src.features import build_features as features

data_dir = Path('../data')
interim_dir = data_dir / 'interim'
processed_dir = data_dir / 'processed'
processed_dir.mkdir(exist_ok=True, parents=True)

print("Loading cohort defined data...")
cohort_df = pd.read_parquet(interim_dir / 'cohort_defined.parquet')
print(f"Loaded {len(cohort_df)} records.")

print("Loading baseline creatinine...")
baseline_cr = pd.read_parquet(interim_dir / 'baseline_creatinine.parquet')
print(f"Loaded {len(baseline_cr)} records.")

In [ ]:
print("Adding missingness indicators...")
df_with_missingness = features.add_missingness_indicators(cohort_df)
print(f"Dataset shape after missingness indicators: {df_with_missingness.shape}")

In [ ]:
print("Adding rolling features...")
df_rolling = features.add_rolling_features(df_with_missingness)
print(f"Dataset shape after rolling features: {df_rolling.shape}")

print("Imputing missing values...")
df_imputed = features.impute_values(df_rolling)
print(f"Dataset shape after imputation: {df_imputed.shape}")

## Creatinine Velocity

Creatinine velocity is perhaps the most important feature for predicting SA-AKI. It captures the **rate of change** of serum creatinine over recent time windows (e.g., 24 hours, 48 hours).

A rapid rise in creatinine is a stronger predictor of imminent kidney injury than the absolute level of creatinine alone, as it directly reflects acute loss of glomerular filtration.

In [ ]:
print("Calculating creatinine velocity...")
df_cr_velocity = features.add_creatinine_velocity(df_imputed, baseline_cr)
print(f"Dataset shape after creatinine velocity: {df_cr_velocity.shape}")

In [ ]:
print("Adding composite features...")
df_composites = features.add_composite_features(df_cr_velocity)
print(f"Dataset shape after composite features: {df_composites.shape}")

In [ ]:
print("Running master feature engineering pipeline...")
final_features_df = features.engineer_features(cohort_df, baseline_cr)
print(f"Final dataset shape: {final_features_df.shape}")

# Save final features
features_path = processed_dir / 'engineered_features.parquet'
final_features_df.to_parquet(features_path)
print(f"Features saved to: {features_path}")

In [ ]:
print("Creating feature manifest...")
manifest = features.create_feature_manifest(final_features_df)
print(f"Generated manifest with {len(manifest)} features.")

import json
with open(processed_dir / 'feature_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=4)

In [ ]:
print("Performing temporal train/val/test split...")
train_df, val_df, test_df = features.temporal_split(final_features_df)

print(f"Train shape: {train_df.shape}")
print(f"Val shape:   {val_df.shape}")
print(f"Test shape:  {test_df.shape}")

train_df.to_parquet(processed_dir / 'train_features.parquet')
val_df.to_parquet(processed_dir / 'val_features.parquet')
test_df.to_parquet(processed_dir / 'test_features.parquet')
print("Data splits saved successfully.")